In [1]:
import pandas as pd
data = pd.read_csv("loan_data_train.csv")

In [2]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2200 entries, 0 to 2199
Data columns (total 15 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   ID                              2199 non-null   float64
 1   Amount.Requested                2199 non-null   object 
 2   Amount.Funded.By.Investors      2199 non-null   object 
 3   Interest.Rate                   2200 non-null   object 
 4   Loan.Length                     2199 non-null   object 
 5   Loan.Purpose                    2199 non-null   object 
 6   Debt.To.Income.Ratio            2199 non-null   object 
 7   State                           2199 non-null   object 
 8   Home.Ownership                  2199 non-null   object 
 9   Monthly.Income                  2197 non-null   float64
 10  FICO.Range                      2200 non-null   object 
 11  Open.CREDIT.Lines               2196 non-null   object 
 12  Revolving.CREDIT.Balance        21

In [3]:
#ignore : id [unique id], Interest.Rate[target]
###info : numeric: data type : object :typecasting
#Amount.requested
#amount.Funded.By.Investors


### info :categorical , create dummies for it
# Loan.length
# FICO.Range : custom function
value = data['Loan.Purpose'].value_counts(dropna = False)
value

Loan.Purpose
debt_consolidation    1147
credit_card            394
other                  174
home_improvement       135
major_purchase          84
small_business          80
car                     45
wedding                 35
medical                 26
moving                  25
house                   19
vacation                18
educational             14
renewable_energy         3
NaN                      1
Name: count, dtype: int64

In [4]:
import ml_utils as mt

In [5]:
custom_func_cols={}

In [6]:
a = data['Debt.To.Income.Ratio']

In [7]:
temp = a.str.replace('%','')

In [8]:
num = pd.to_numeric(temp,errors='coerce')
num

0       27.56
1       13.39
2        3.50
3       19.62
4       23.79
        ...  
2195    12.10
2196    14.16
2197    15.03
2198    11.63
2199     3.83
Name: Debt.To.Income.Ratio, Length: 2200, dtype: float64

In [9]:
def custom_dir(dir_col):
    temp = a.str.replace('%','')
    num = pd.to_numeric(temp,errors='coerce')
    return num

In [10]:
custom_dir(data['Debt.To.Income.Ratio'])

0       27.56
1       13.39
2        3.50
3       19.62
4       23.79
        ...  
2195    12.10
2196    14.16
2197    15.03
2198    11.63
2199     3.83
Name: Debt.To.Income.Ratio, Length: 2200, dtype: float64

In [11]:
b = data['FICO.Range']

In [12]:
def custom_FICO(b):
    k = b.str.split('-',expand = True)
    for i in [0,1]:
        k[i]= pd.to_numeric(k[i],errors='coerce')
    n = 0.5*(k[0]+k[1])
    return n

In [13]:
custom_FICO(data['FICO.Range'])

0       722.0
1       712.0
2       692.0
3       712.0
4       732.0
        ...  
2195    677.0
2196    702.0
2197    677.0
2198    672.0
2199    712.0
Length: 2200, dtype: float64

In [14]:
a= data['Employment.Length']

In [15]:
def custom_el(el_col):
    temp=el_col.replace({'5 years':5,'4 years':4,'<1 year':0,'10+ years':10,'2 years':2,'3 years':3,'1 year':1,'6 years':6,'7 years':7,'8 years':8,'9 years':9})
    num = pd.to_numeric(temp,errors='coerce')
    return num

In [16]:
cat_to_num_cols=['Amount.Requested','Amount.Funded.By.Investors','Open.CREDIT.Lines','Revolving.CREDIT.Balance']
simple_num_cols=['Monthly.Income','Inquiries.in.the.Last.6.Months']
cat_to_dummies_cols=['Loan.Length','Loan.Purpose','State','Home.Ownership']
cust_func_cols={'Debt.To.Income.Ratio':custom_dir,'FICO.Range':custom_FICO,'Employment.Length':custom_el}

In [17]:
ld_pipe=mt.DataPipe(simple_num=simple_num_cols,cat_to_dummies=cat_to_dummies_cols,cat_to_num=cat_to_num_cols,custom_func_dict=custom_func_cols)

In [18]:
ld_pipe.fit(data)

d:\prog\Python(Narind Verma)\WORKSHOP\w1\ml_utils.py:75: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x[col]=pd.to_numeric(x[col],errors='coerce')


In [19]:
x_train = ld_pipe.transform(data)


d:\prog\Python(Narind Verma)\WORKSHOP\w1\ml_utils.py:75: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x[col]=pd.to_numeric(x[col],errors='coerce')


In [20]:
x_train.shape

(2200, 46)

In [21]:
ld_test = pd.read_csv('./loan_data_test.csv')

In [22]:
x_test=ld_pipe.transform(ld_test)

d:\prog\Python(Narind Verma)\WORKSHOP\w1\ml_utils.py:75: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x[col]=pd.to_numeric(x[col],errors='coerce')


In [23]:
x_test.to_csv("test_transformed.csv")